In [1]:
# -*- coding: utf-8 -*-
"""
================================================================================
Binary Diabetic Retinopathy Detection — PATIENT-LEVEL PREPROCESSING
================================================================================

This script creates patient-level labels and balanced train/val/test CSV splits
for the EyePACS dataset.

What this script does:
1. Load and process labels at patient level
2. Split patients into train/val/test (70/15/15) with stratification
3. Balance training patients only via undersampling
4. Expand patient splits to image-level CSVs
5. Save retinal_train_images.csv, retinal_val_images.csv, retinal_test_images.csv

Note: Image preprocessing, transforms, and PyTorch Dataset are in the training script.

Author: [Your Name]
Date: 2025
Project: Multimodal Federated Learning for Diabetes Detection
================================================================================
"""

# ============================================================
# 1. Imports and Configuration
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample


class Config:
    """Configuration for retinal data preprocessing."""

    # Data paths
    DATA_DIR = "/content/drive/MyDrive/MULTIMODAL/datasets/EYEPACS/images/resized_train_cropped"
    LABELS_PATH = "/content/drive/MyDrive/MULTIMODAL/datasets/EYEPACS/trainLabels_cropped.csv"
    OUTPUT_DIR = "/content/drive/MyDrive/MULTIMODAL/datasets/EYEPACS/splits"

    # Split configuration
    TEST_SIZE = 0.15
    VAL_SIZE = 0.15
    # Results in: Train 70%, Val 15%, Test 15%

    # Balancing
    BALANCE_STRATEGY = 'undersample'  # Options: 'undersample', 'oversample'
    BALANCE_TRAIN_ONLY = True

    # Reproducibility
    SEED = 42


def seed_everything(seed: int = 42):
    """Set random seeds for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    print(f"[INFO] Seed set to {seed}")


# ============================================================
# 2. Patient-Level Label Processing
# ============================================================

def extract_patient_id(image_name: str) -> str:
    """Extract patient ID from image filename (e.g., '10_left' -> '10')."""
    return image_name.rsplit('_', 1)[0]


def extract_eye_side(image_name: str) -> str:
    """Extract eye side from image filename (e.g., '10_left' -> 'left')."""
    return image_name.rsplit('_', 1)[1]


def load_and_process_labels_patient_level(labels_path: str):
    """
    Load labels and create patient-level labels.

    Patient is labeled as diabetic (1) if EITHER eye has level >= 1.

    Parameters
    ----------
    labels_path : str
        Path to the CSV file with image-level labels.

    Returns
    -------
    image_df : pd.DataFrame
        Image-level dataframe with patient labels added.
    patient_df : pd.DataFrame
        Patient-level dataframe with aggregated labels.
    """
    # Load raw labels
    df = pd.read_csv(labels_path)
    print(f"\n[INFO] Loaded labels: {len(df)} images")

    # Extract patient ID and eye side
    df['patient_id'] = df['image'].apply(extract_patient_id)
    df['eye'] = df['image'].apply(extract_eye_side)

    # Create binary label for each image (level >= 1 means diabetic)
    df['binary_label'] = (df['level'] >= 1).astype(int)

    # Create patient-level pivot table
    pivot = df.pivot_table(
        index='patient_id',
        columns='eye',
        values='level',
        aggfunc='first'
    ).reset_index()
    pivot.columns.name = None
    pivot = pivot.rename(columns={'left': 'left_level', 'right': 'right_level'})

    # Fill missing eyes with -1 (indicates missing)
    pivot['left_level'] = pivot['left_level'].fillna(-1).astype(int)
    pivot['right_level'] = pivot['right_level'].fillna(-1).astype(int)

    # Create patient-level label: 1 if either eye has level >= 1
    def compute_patient_label(row):
        left = row['left_level'] if row['left_level'] >= 0 else 0
        right = row['right_level'] if row['right_level'] >= 0 else 0
        return 1 if (left >= 1 or right >= 1) else 0

    pivot['patient_label'] = pivot.apply(compute_patient_label, axis=1)

    # Merge patient labels back to image dataframe
    df = df.merge(pivot[['patient_id', 'patient_label']], on='patient_id', how='left')

    # Print statistics
    print(f"[INFO] Unique patients: {len(pivot)}")
    print(f"[INFO] Patient label distribution:")
    print(pivot['patient_label'].value_counts().to_string())

    # Select relevant columns for image dataframe
    image_df = df[['image', 'patient_id', 'eye', 'level', 'binary_label', 'patient_label']].copy()
    patient_df = pivot

    return image_df, patient_df


# ============================================================
# 3. Patient-Level Split and Balance Utilities
# ============================================================

def split_patients(patient_df: pd.DataFrame, test_size: float = 0.15,
                   val_size: float = 0.15, seed: int = 42):
    """
    Split patients into train/val/test sets with stratification.

    Parameters
    ----------
    patient_df : pd.DataFrame
        Patient-level dataframe with 'patient_label' column.
    test_size : float
        Fraction for test set.
    val_size : float
        Fraction for validation set.
    seed : int
        Random seed.

    Returns
    -------
    train_patients, val_patients, test_patients : pd.DataFrame
        Patient dataframes for each split.
    """
    # First split: separate test set
    train_val, test = train_test_split(
        patient_df,
        test_size=test_size,
        stratify=patient_df['patient_label'],
        random_state=seed
    )

    # Second split: separate train and validation
    val_ratio = val_size / (1 - test_size)
    train, val = train_test_split(
        train_val,
        test_size=val_ratio,
        stratify=train_val['patient_label'],
        random_state=seed
    )

    print(f"\n[INFO] Patient splits:")
    print(f"  Train: {len(train)} patients")
    print(f"  Val:   {len(val)} patients")
    print(f"  Test:  {len(test)} patients")

    return train, val, test


def balance_patients(df: pd.DataFrame, strategy: str = 'undersample', seed: int = 42):
    """
    Balance patient classes via undersampling or oversampling.

    Parameters
    ----------
    df : pd.DataFrame
        Patient dataframe with 'patient_label' column.
    strategy : str
        'undersample' - reduce majority class to match minority
        'oversample' - increase minority class to match majority
    seed : int
        Random seed.

    Returns
    -------
    pd.DataFrame
        Balanced patient dataframe.
    """
    class_0 = df[df['patient_label'] == 0]
    class_1 = df[df['patient_label'] == 1]

    print(f"\n[INFO] Before balancing:")
    print(f"  Class 0 (No DR): {len(class_0)}")
    print(f"  Class 1 (DR):    {len(class_1)}")

    if strategy == 'undersample':
        n_samples = min(len(class_0), len(class_1))
        class_0_balanced = resample(class_0, n_samples=n_samples,
                                     random_state=seed, replace=False)
        class_1_balanced = resample(class_1, n_samples=n_samples,
                                     random_state=seed, replace=False)
    elif strategy == 'oversample':
        n_samples = max(len(class_0), len(class_1))
        class_0_balanced = resample(class_0, n_samples=n_samples,
                                     random_state=seed, replace=True)
        class_1_balanced = resample(class_1, n_samples=n_samples,
                                     random_state=seed, replace=True)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # Combine and shuffle
    balanced_df = pd.concat([class_0_balanced, class_1_balanced])
    balanced_df = balanced_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\n[INFO] After balancing ({strategy}):")
    print(f"  Class 0 (No DR): {len(class_0_balanced)}")
    print(f"  Class 1 (DR):    {len(class_1_balanced)}")
    print(f"  Total patients:  {len(balanced_df)}")

    return balanced_df


def expand_patients_to_images(patient_df: pd.DataFrame, image_df: pd.DataFrame):
    """
    Expand patient-level split to image-level by selecting all images
    belonging to the patients in the split.

    Parameters
    ----------
    patient_df : pd.DataFrame
        Patient-level dataframe with 'patient_id' column.
    image_df : pd.DataFrame
        Full image-level dataframe.

    Returns
    -------
    pd.DataFrame
        Image-level dataframe for the selected patients.
    """
    patient_ids = patient_df['patient_id'].unique()
    expanded = image_df[image_df['patient_id'].isin(patient_ids)].copy()
    return expanded


def print_split_statistics_patient_level(train_images, val_images, test_images,
                                          train_patients, val_patients, test_patients):
    """Print detailed statistics for each split."""

    print("\n" + "=" * 60)
    print("SPLIT STATISTICS")
    print("=" * 60)

    for name, imgs, pats in [('Train', train_images, train_patients),
                              ('Val', val_images, val_patients),
                              ('Test', test_images, test_patients)]:
        print(f"\n{name} Set:")
        print(f"  Patients: {len(pats)}")
        print(f"  Images:   {len(imgs)}")

        # Patient-level distribution
        pat_dist = pats['patient_label'].value_counts().sort_index()
        print(f"  Patient distribution: {pat_dist.to_dict()}")

        # Image-level distribution
        img_dist = imgs['patient_label'].value_counts().sort_index()
        print(f"  Image distribution:   {img_dist.to_dict()}")

        # Calculate ratios
        if len(pat_dist) == 2:
            ratio = pat_dist.iloc[0] / pat_dist.iloc[1] if pat_dist.iloc[1] > 0 else float('inf')
            print(f"  Class ratio (0:1):    {ratio:.2f}")


# ============================================================
# 4. Main Execution — Create and Save CSV Splits
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("RETINAL DATA PREPROCESSING — PATIENT-LEVEL SPLITS")
    print("=" * 60)

    # Initialize
    config = Config()
    seed_everything(config.SEED)

    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    # ─────────────────────────────────────────────────────────
    # Step 1: Load and process labels
    # ─────────────────────────────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 1: Loading and processing labels")
    print("-" * 40)

    image_df, patient_df = load_and_process_labels_patient_level(config.LABELS_PATH)

    # ─────────────────────────────────────────────────────────
    # Step 2: Split patients into train/val/test
    # ─────────────────────────────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 2: Splitting patients (stratified)")
    print("-" * 40)

    train_patients, val_patients, test_patients = split_patients(
        patient_df,
        test_size=config.TEST_SIZE,
        val_size=config.VAL_SIZE,
        seed=config.SEED
    )

    # ─────────────────────────────────────────────────────────
    # Step 3: Balance training patients only
    # ─────────────────────────────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 3: Balancing training patients")
    print("-" * 40)

    if config.BALANCE_TRAIN_ONLY:
        train_patients_balanced = balance_patients(
            train_patients,
            strategy=config.BALANCE_STRATEGY,
            seed=config.SEED
        )
    else:
        train_patients_balanced = train_patients
        print("[INFO] Skipping balancing (BALANCE_TRAIN_ONLY=False)")

    # ─────────────────────────────────────────────────────────
    # Step 4: Expand patient splits to image-level
    # ─────────────────────────────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 4: Expanding to image-level splits")
    print("-" * 40)

    train_images = expand_patients_to_images(train_patients_balanced, image_df)
    val_images = expand_patients_to_images(val_patients, image_df)
    test_images = expand_patients_to_images(test_patients, image_df)

    print(f"  Train images: {len(train_images)}")
    print(f"  Val images:   {len(val_images)}")
    print(f"  Test images:  {len(test_images)}")

    # ─────────────────────────────────────────────────────────
    # Step 5: Print statistics
    # ─────────────────────────────────────────────────────────
    print_split_statistics_patient_level(
        train_images, val_images, test_images,
        train_patients_balanced, val_patients, test_patients
    )

    # ─────────────────────────────────────────────────────────
    # Step 6: Save CSV splits
    # ─────────────────────────────────────────────────────────
    print("\n" + "-" * 40)
    print("STEP 5: Saving CSV splits")
    print("-" * 40)

    train_path = os.path.join(config.OUTPUT_DIR, "retinal_train_images.csv")
    val_path = os.path.join(config.OUTPUT_DIR, "retinal_val_images.csv")
    test_path = os.path.join(config.OUTPUT_DIR, "retinal_test_images.csv")

    train_images.to_csv(train_path, index=False)
    val_images.to_csv(val_path, index=False)
    test_images.to_csv(test_path, index=False)

    print(f"  ✓ Saved: {train_path}")
    print(f"  ✓ Saved: {val_path}")
    print(f"  ✓ Saved: {test_path}")

    # ─────────────────────────────────────────────────────────
    # Summary
    # ─────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("PREPROCESSING COMPLETE")
    print("=" * 60)

    print(f"""
Summary:
────────────────────────────────────────
  Total patients:     {len(patient_df)}
  Total images:       {len(image_df)}

  Train (balanced):   {len(train_patients_balanced)} patients, {len(train_images)} images
  Validation:         {len(val_patients)} patients, {len(val_images)} images
  Test:               {len(test_patients)} patients, {len(test_images)} images

  Balancing strategy: {config.BALANCE_STRATEGY}
  Balance train only: {config.BALANCE_TRAIN_ONLY}

  Output directory:   {config.OUTPUT_DIR}
────────────────────────────────────────

Next step: Use these CSVs in the training script with image preprocessing.
""")

RETINAL DATA PREPROCESSING — PATIENT-LEVEL SPLITS
[INFO] Seed set to 42

----------------------------------------
STEP 1: Loading and processing labels
----------------------------------------

[INFO] Loaded labels: 35108 images
[INFO] Unique patients: 17561
[INFO] Patient label distribution:
patient_label
0    12155
1     5406

----------------------------------------
STEP 2: Splitting patients (stratified)
----------------------------------------

[INFO] Patient splits:
  Train: 12292 patients
  Val:   2634 patients
  Test:  2635 patients

----------------------------------------
STEP 3: Balancing training patients
----------------------------------------

[INFO] Before balancing:
  Class 0 (No DR): 8508
  Class 1 (DR):    3784

[INFO] After balancing (undersample):
  Class 0 (No DR): 3784
  Class 1 (DR):    3784
  Total patients:  7568

----------------------------------------
STEP 4: Expanding to image-level splits
----------------------------------------
  Train images: 15128
  Va